In [ ]:
!pip install -q -U transformers accelerate datasets huggingface_hub pandas pyarrow
!pip install -q sae-lens

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# import sys
# sys.path.insert(1, "/kaggle/input/corpus_loader")

import time
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from sae_lens import SAE

from corpus_loader import CorpusConfig, chunk_stream, batch_chunks

In [ ]:
# ---- модель ----
MODEL_ID = "google/gemma-2-2b"
LAYER_IDX = 12
DTYPE = torch.float16

SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER_IDX}/width_16k/canonical"
SAE_WIDTH = "16k"

# ---- отобранные фичи ----
BIMODAL_FEATURES = [10301, 2241, 11343, 12428, 1673]

# Контроль: унимодальные хвосты (dip_p > 0.998) с СОПОСТАВИМОЙ density
CONTROL_FEATURES = [6986, 4350, 1809, 5, 7911]

FEATURES = sorted(set(BIMODAL_FEATURES + CONTROL_FEATURES))
print(f"Следим за {len(FEATURES)} фичами: {FEATURES}")

# ---- корпус ----
SEQ_LEN = 1024
TARGET_TOKENS = 10_000_000
BATCH_SIZE = 8
SAE_CHUNK = 1024

# ---- контекст ----
CTX_BEFORE = 30
CTX_AFTER = 10

# ---- инфраструктура ----
SECRET_NAME = "llama-token"
OUTPUT_DIR = "/kaggle/working"
OUT_PATH = os.path.join(OUTPUT_DIR, f"pass2_contexts_{SAE_WIDTH}.parquet")
FLUSH_EVERY = 50_000           
MAX_RUNTIME_HOURS = 8.0

In [ ]:
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret(SECRET_NAME)
    print("secret ok:", hf_token[:7] + "...")
except Exception as e:
    print("Секрет не получен:", repr(e))

if hf_token:
    os.environ["HF_TOKEN"] = hf_token

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map={"": 0})
model.eval()

device = next(model.parameters()).device
print("device:", device, "| dtype:", next(model.parameters()).dtype)
print("devices:", set(str(p.device) for p in model.parameters()))
print("n_layers:", len(model.layers), "| d_model:", model.config.hidden_size)

In [ ]:
out = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=str(device))
sae = out[0] if isinstance(out, tuple) else out
sae.eval()

d_sae, d_model = sae.W_dec.shape
assert d_model == model.config.hidden_size, (
    f"W_dec.shape={tuple(sae.W_dec.shape)}, ожидали (d_sae, {model.config.hidden_size})"
)
print(f"SAE: d_sae={d_sae}, d_model={d_model}")

assert max(FEATURES) < d_sae, f"feature id {max(FEATURES)} вне словаря размера {d_sae}"

feat_tensor = torch.tensor(FEATURES, device=device)
print("OK, все feature id валидны")

In [ ]:
_captured = {}

def _hook_fn(module, inputs, output):
    _captured["hidden"] = output[0] if isinstance(output, tuple) else output

target_module = model.layers[LAYER_IDX]
handle = target_module.register_forward_hook(_hook_fn)
print(f"Hook на model.layers[{LAYER_IDX}] (residual stream, post-block)")

In [ ]:
import glob

records = []          
n_flushed = 0
_part = 0
PART_GLOB = os.path.join(OUTPUT_DIR, "pass2_part_*.parquet")

for p in glob.glob(PART_GLOB):
    os.remove(p)

def flush(force=False):
    '''Сбрасывает буфер в отдельный part-файл.
    Не перечитываем накопленное (это было бы O(n^2) по времени) — части
    склеиваются один раз в конце.'''
    global records, n_flushed, _part
    if not records or (len(records) < FLUSH_EVERY and not force):
        return
    path = os.path.join(OUTPUT_DIR, f"pass2_part_{_part:04d}.parquet")
    pd.DataFrame(records).to_parquet(path, index=False)
    n_flushed += len(records)
    _part += 1
    records = []
    print(f"  [flush] part {_part}, всего записей: {n_flushed:,}")

In [ ]:
cfg = CorpusConfig(seq_len=SEQ_LEN, skip_tokens=0, max_tokens=TARGET_TOKENS, seed=0)
batches = batch_chunks(chunk_stream(tokenizer, cfg), BATCH_SIZE)

start = time.time()
tokens_seen = 0
n_batches = 0

with torch.no_grad():
    for batch in batches:
        elapsed_h = (time.time() - start) / 3600
        if elapsed_h > MAX_RUNTIME_HOURS:
            print("Бюджет времени исчерпан.")
            break

        batch_gpu = batch.to(device)
        model(batch_gpu)

        hidden = _captured["hidden"]                   
        B, T, _ = hidden.shape
        hidden_flat = hidden.reshape(-1, hidden.shape[-1])

        batch_cpu = batch.numpy()                      

        for i in range(0, hidden_flat.shape[0], SAE_CHUNK):
            sub = hidden_flat[i:i + SAE_CHUNK].to(sae.W_enc.dtype)
            z = sae.encode(sub)                         
            z_sub = z[:, feat_tensor]                  

            rows, cols = z_sub.nonzero(as_tuple=True)
            if rows.numel():
                vals = z_sub[rows, cols].float().cpu().numpy()
                rows = rows.cpu().numpy()
                cols = cols.cpu().numpy()
                for row, col, val in zip(rows, cols, vals):
                    flat_pos = i + int(row)            
                    b = flat_pos // T
                    t = flat_pos % T
                    fid = FEATURES[int(col)]

                    lo = max(0, t - CTX_BEFORE)
                    hi = min(T, t + CTX_AFTER + 1)
                    ctx_ids = batch_cpu[b, lo:hi]

                    records.append({
                        "feature": fid,
                        "value": float(val),
                        "token_id": int(batch_cpu[b, t]),
                        "token": tokenizer.decode([int(batch_cpu[b, t])]),
                        "context": tokenizer.decode(ctx_ids),
                        "target_pos_in_ctx": int(t - lo),
                        "global_chunk": n_batches * BATCH_SIZE + b,
                        "pos_in_chunk": int(t),
                    })
            del z, z_sub, sub

        flush()

        del hidden, hidden_flat
        _captured.clear()

        tokens_seen += B * T
        n_batches += 1

        if n_batches % 50 == 0:
            rate = tokens_seen / max(time.time() - start, 1)
            print(f"tokens={tokens_seen:,} | буфер={len(records):,} | "
                  f"{elapsed_h:.2f}ч | ~{rate:.0f} tok/s")

flush(force=True)

parts = sorted(glob.glob(PART_GLOB))
if parts:
    pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True).to_parquet(OUT_PATH, index=False)
    for p in parts:
        os.remove(p)
    print(f"Склеено {len(parts)} частей -> {OUT_PATH}")
else:
    print("ВНИМАНИЕ: ни одной записи не собрано!")

print(f"\nГотово. tokens_seen={tokens_seen:,}, записей={n_flushed:,}")

handle.remove()

## Проверка собранного

In [ ]:
df = pd.read_parquet(OUT_PATH)
print(f"Всего записей: {len(df):,}\n")

stat = df.groupby("feature")["value"].agg(["count", "min", "median", "max"])
stat["density"] = stat["count"] / max(tokens_seen, 1)
print(stat.to_string())

print("\nЕсли у какой-то фичи count < 500 — данных мало для разбора по пикам, "
      "нужен более длинный прогон.")

In [ ]:
# Это sanity-check: если форма другая, значит что-то в сетапе разъехалось.
import matplotlib.pyplot as plt

feats = sorted(df.feature.unique())
n = len(feats)
fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(15, 4 * ((n + 2) // 3)))
axes = np.atleast_1d(axes).ravel()

for ax, f in zip(axes, feats):
    v = df[df.feature == f].value.values
    ax.hist(np.log10(v), bins=50)
    ax.set_title(f"#{f} | n={len(v):,}")
    ax.set_xlabel("log10(activation)")

for ax in axes[len(feats):]:
    ax.axis("off")

plt.tight_layout()
plt.show()